# 05 — Exploratory Analysis

**Purpose:** Visualize data quality, feature distributions, and group differences between High/Low performers to guide modeling decisions.

**Inputs:**
- `data/processed/dataset_final.parquet`
- `data/processed/labels.csv`
- `data/processed/raw_gaze_clean.parquet`

**Outputs:** Figures saved to `outputs/figures/`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

from src.config import (
    DATASET_FINAL, LABELS_CSV, RAWGAZE_CLEAN_PKL, OUTPUTS_FIGURES,
    PERFORMANCE_LABEL_COL, TASKS, ExportCols
)

sns.set_theme(style='whitegrid', palette='Set2')
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)

LABEL_COLORS = {1: '#2ecc71', 0: '#e74c3c'}
LABEL_NAMES  = {1: 'High', 0: 'Low'}

## 1. Load data

In [ ]:
dataset = pd.read_parquet(DATASET_FINAL)
labels  = pd.read_csv(LABELS_CSV)
gaze    = pd.read_parquet(RAWGAZE_CLEAN_PKL)

# Use unscaled features for EDA
unscaled_cols = [c for c in dataset.columns if c.endswith('_unscaled')]
meta_cols = ['participant_id', PERFORMANCE_LABEL_COL, 'speed_label', 'accuracy_rate',
             'total_score', 'split']
meta_cols = [c for c in meta_cols if c in dataset.columns]

df = dataset[meta_cols + unscaled_cols].copy()
# Strip '_unscaled' suffix for display
df.columns = [c.replace('_unscaled', '') for c in df.columns]

feature_cols = [c for c in df.columns if c not in meta_cols]
print(f"Participants: {len(df)}, Feature columns: {len(feature_cols)}")

## 2. Class distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Class counts
counts = df[PERFORMANCE_LABEL_COL].map(LABEL_NAMES).value_counts()
colors = [LABEL_COLORS[k] for k in df[PERFORMANCE_LABEL_COL].map(LABEL_NAMES).value_counts().index.map({v: k for k, v in LABEL_NAMES.items()})]
counts.plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'][:len(counts)], edgecolor='white')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Accuracy histogram by group
for label, grp in df.groupby(PERFORMANCE_LABEL_COL):
    axes[1].hist(grp['accuracy_rate'], bins=8, alpha=0.6,
                 color=LABEL_COLORS[label], label=LABEL_NAMES[label], edgecolor='white')
axes[1].set_title('Accuracy Rate by Group')
axes[1].set_xlabel('Accuracy rate')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '05_class_distribution.png', dpi=150)
plt.show()

## 3. Violin plots of key features by group

In [ ]:
# Select key features for display (use aggregate/cross-task features if available)
key_features = [c for c in [
    'mean_fixation_duration_across_tasks',
    'total_correct_aoi_dwell_ratio',
    'mean_time_to_first_click',
    'mean_distractor_dwell_ratio',
] if c in df.columns]

# Also try per-task features for a specific task
sample_task_feats = [c for c in feature_cols if 'findDice' in c and 'correct' in c][:4]
display_features = key_features + sample_task_feats

if not display_features:
    display_features = feature_cols[:8]

n = len(display_features)
cols_per_row = 4
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(cols_per_row * 4, rows * 4))
axes = np.array(axes).flatten()

plot_df = df[[PERFORMANCE_LABEL_COL] + display_features].copy()
plot_df[PERFORMANCE_LABEL_COL] = plot_df[PERFORMANCE_LABEL_COL].map(LABEL_NAMES)

for i, feat in enumerate(display_features):
    sns.violinplot(
        data=plot_df, x=PERFORMANCE_LABEL_COL, y=feat,
        palette=list(LABEL_COLORS.values()), ax=axes[i], inner='box'
    )
    axes[i].set_title(feat.replace('_', '\n'), fontsize=8)
    axes[i].set_xlabel('')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by Performance Group', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '05_violin_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correct AOI dwell ratio per task — main interpretive plot

In [ ]:
dwell_cols = [c for c in df.columns if 'correct_aoi_dwell_ratio' in c]

if dwell_cols:
    dwell_data = []
    for col in dwell_cols:
        task_name = col.split('_correct_aoi')[0]
        for label, grp in df.groupby(PERFORMANCE_LABEL_COL):
            dwell_data.append({
                'task': task_name,
                'group': LABEL_NAMES[label],
                'mean_dwell': grp[col].mean()
            })

    dwell_df = pd.DataFrame(dwell_data)

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(dwell_cols))
    width = 0.35
    task_labels = [c.split('_correct_aoi')[0] for c in dwell_cols]

    for i, (label, group_name, color) in enumerate([(1,'High','#2ecc71'), (0,'Low','#e74c3c')]):
        grp_data = dwell_df[dwell_df['group'] == group_name]['mean_dwell'].values
        ax.bar(x + i*width, grp_data[:len(x)], width, label=group_name, color=color, alpha=0.85)

    ax.set_xticks(x + width/2)
    ax.set_xticklabels(task_labels, rotation=30, ha='right')
    ax.set_ylabel('Mean Correct AOI Dwell Ratio')
    ax.set_title('Correct AOI Dwell Ratio per Task: High vs Low Performers')
    ax.legend()
    ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(OUTPUTS_FIGURES / '05_dwell_ratio_per_task.png', dpi=150)
    plt.show()
else:
    print("No correct AOI dwell ratio columns found.")

## 5. Scanpath visualization (one High vs one Low participant)

In [ ]:
# Select representative participants
high_ids = df[df[PERFORMANCE_LABEL_COL] == 1]['participant_id'].values
low_ids  = df[df[PERFORMANCE_LABEL_COL] == 0]['participant_id'].values

SAMPLE_TASK = TASKS[0]  # Use first task for visualization

def plot_scanpath(ax, participant_id, task, gaze_df, title, color):
    p_gaze = gaze_df[
        (gaze_df[ExportCols.PARTICIPANT_NAME] == participant_id) &
        (gaze_df[ExportCols.PRESENTED_MEDIA].str.contains(task, na=False, case=False)) &
        (gaze_df[ExportCols.EYE_MOVEMENT_TYPE] == 'Fixation')
    ].dropna(subset=[ExportCols.FIXATION_X, ExportCols.FIXATION_Y, ExportCols.EYE_MOVEMENT_DURATION])

    if len(p_gaze) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return

    # Normalize duration for circle sizes
    sizes = (p_gaze[ExportCols.EYE_MOVEMENT_DURATION] / p_gaze[ExportCols.EYE_MOVEMENT_DURATION].max() * 500).clip(20, 500)

    ax.scatter(
        p_gaze[ExportCols.FIXATION_X],
        p_gaze[ExportCols.FIXATION_Y],
        s=sizes, c=color, alpha=0.6, edgecolors='white', linewidths=0.5
    )
    # Draw scanpath lines
    ax.plot(
        p_gaze[ExportCols.FIXATION_X].values,
        p_gaze[ExportCols.FIXATION_Y].values,
        color=color, alpha=0.3, linewidth=1
    )
    ax.invert_yaxis()  # screen coordinates: Y increases downward
    ax.set_title(f"{title}\n({len(p_gaze)} fixations)", fontsize=10)
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(high_ids) > 0:
    plot_scanpath(axes[0], high_ids[0], SAMPLE_TASK, gaze, f'High Performer\n({SAMPLE_TASK})', '#2ecc71')
if len(low_ids) > 0:
    plot_scanpath(axes[1], low_ids[0], SAMPLE_TASK, gaze, f'Low Performer\n({SAMPLE_TASK})', '#e74c3c')

plt.suptitle('Scanpath Comparison: High vs Low Performer', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '05_scanpath_comparison.png', dpi=150)
plt.show()

## 6. Correlation heatmap

In [ ]:
# Select top 20 most variable features
top_feats = df[feature_cols].std().nlargest(20).index.tolist()

corr = df[top_feats].corr(method='spearman')

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr, ax=ax,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 6},
    linewidths=0.5
)
ax.set_title('Spearman Correlation — Top 20 Most Variable Features')
plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Mann-Whitney U test per feature

In [ ]:
high_group = df[df[PERFORMANCE_LABEL_COL] == 1]
low_group  = df[df[PERFORMANCE_LABEL_COL] == 0]

test_results = []
for feat in feature_cols:
    h = high_group[feat].dropna()
    l = low_group[feat].dropna()
    if len(h) < 3 or len(l) < 3:
        continue
    u_stat, p_val = stats.mannwhitneyu(h, l, alternative='two-sided')

    # Rank-biserial correlation (effect size)
    n1, n2 = len(h), len(l)
    r = 1 - (2 * u_stat) / (n1 * n2)

    test_results.append({
        'feature': feat,
        'U_stat': u_stat,
        'p_value': p_val,
        'effect_size_r': abs(r),
        'high_mean': h.mean(),
        'low_mean': l.mean(),
    })

results_df = pd.DataFrame(test_results).sort_values('p_value')

# FDR correction
reject, p_corrected, _, _ = multipletests(results_df['p_value'], method='fdr_bh')
results_df['p_fdr'] = p_corrected
results_df['significant_fdr'] = reject

print(f"Features significant before correction (p<0.05): {(results_df['p_value'] < 0.05).sum()}")
print(f"Features significant after FDR correction:       {reject.sum()}")
results_df.head(20)

In [ ]:
# Plot top features by effect size
top_sig = results_df.nlargest(15, 'effect_size_r')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if sig else '#bdc3c7' for sig in top_sig['significant_fdr']]
ax.barh(top_sig['feature'].str.replace('_', ' '), top_sig['effect_size_r'],
        color=colors, edgecolor='white')
ax.set_xlabel('Effect size |r| (rank-biserial)')
ax.set_title('Top Features by Effect Size (green = FDR significant)')
ax.axvline(0.3, color='gray', linestyle='--', label='Medium effect (r=0.3)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '05_effect_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
results_df.to_csv('../outputs/reports/05_mann_whitney_results.csv', index=False)
print("Saved statistical test results.")